# DGX Testing: U-Net Melting-Layer Detector

This notebook evaluates a trained U-Net model on the test split across ~98 GPM DPR HDF5 files on a DGX GPU, computes metrics with a threshold sweep, exports CSV summaries, renders qualitative figures, and benchmarks throughput.

In [ ]:
# GPU setup (memory growth, mixed precision, strategy)
import os, random, json, time
import numpy as np

# Optional: XLA
os.environ.setdefault('TF_XLA_FLAGS', '--tf_xla_auto_jit=2')

import tensorflow as tf
from tensorflow.keras import mixed_precision

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs visible: {len(gpus)} ->", gpus)
except Exception as e:
    print("GPU setup warning:", e)

mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision policy:", mixed_precision.global_policy())

strategy = tf.distribute.MirroredStrategy()
print("Strategy devices:", strategy.num_replicas_in_sync)

In [ ]:
# Load trained model and config/split
import json, os
import dgx_ml as dgx

CONFIG_JSON = 'config.json'
SPLIT_JSON = 'split.json'
FIG_DIR = 'figures'
RESULTS_DIR = 'results'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(CONFIG_JSON, 'r', encoding='utf-8') as f:
    cfg = json.load(f)
with open(SPLIT_JSON, 'r', encoding='utf-8') as f:
    split = json.load(f)

MASK_MODE = cfg['MASK_MODE']
USE_DFR_ML = cfg['USE_DFR_ML']
INCLUDE_LATLON = cfg['INCLUDE_LATLON']
USE_Z_FINAL = cfg['USE_Z_FINAL']
N_BIN_TARGET = cfg['N_BIN_TARGET']
MIN_VAL, MAX_VAL = cfg['MIN_VAL'], cfg['MAX_VAL']
INFER_THRESHOLD = cfg.get('INFER_THRESHOLD', 0.45)

INPUT_CHANNELS = 4 if INCLUDE_LATLON else 2

# Prefer best checkpoint if available; else SavedModel
best_h5 = cfg.get('BEST_H5')
saved_model_path = cfg.get('SAVEDMODEL_PATH')

model = None
try:
    if best_h5 and os.path.exists(best_h5):
        model = tf.keras.models.load_model(best_h5)
        print('Loaded model from', best_h5)
except Exception as e:
    print('H5 load failed:', e)

if model is None:
    model = tf.keras.models.load_model(saved_model_path)
    print('Loaded model from', saved_model_path)

test_files = split['test']
print('Test files:', len(test_files))

In [ ]:
# Build evaluation dataset (same preprocessing) for test split
import numpy as np

def sample_generator(files,
                     mask_mode=MASK_MODE,
                     use_z_final=USE_Z_FINAL,
                     use_dfr_ml=USE_DFR_ML,
                     include_latlon=INCLUDE_LATLON,
                     nbin_target=N_BIN_TARGET,
                     min_val=MIN_VAL,
                     max_val=MAX_VAL):
    for fp in files:
        X, Y = dgx.build_arrays_for_mask_mode([fp],
                                              mask_mode=mask_mode,
                                              use_z_final=use_z_final,
                                              use_dfr_ml=use_dfr_ml,
                                              include_latlon=include_latlon,
                                              nbin_target=nbin_target,
                                              min_val=min_val,
                                              max_val=max_val,
                                              max_scans=None)
        for i in range(X.shape[0]):
            yield X[i].astype(np.float32), Y[i].astype(np.float32), fp

AUTOTUNE = tf.data.AUTOTUNE

output_signature = (
    tf.TensorSpec(shape=(N_BIN_TARGET, None, INPUT_CHANNELS), dtype=tf.float32),
    tf.TensorSpec(shape=(N_BIN_TARGET, None, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.string)
)

def make_test_dataset(file_list, batch_size=8):
    def gen():
        for x, y, fp in sample_generator(file_list):
            yield x, y, tf.convert_to_tensor(fp)
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

test_ds = make_test_dataset(test_files, batch_size=cfg['BATCH_SIZE'])
print(test_ds)

In [ ]:
# Batched inference and optional per-scan prob saving
import time, os

store_probs = False  # set True to save per-scan .npy (large)
probs_dir = os.path.join(RESULTS_DIR, 'probs')
if store_probs:
    os.makedirs(probs_dir, exist_ok=True)

all_probs = []
all_trues = []
all_files_for_sample = []

start = time.time()
for xb, yb, fb in test_ds:
    yp = model.predict(xb, verbose=0)
    all_probs.append(yp.numpy().astype(np.float32))
    all_trues.append(yb.numpy().astype(np.float32))
    all_files_for_sample += [f.decode('utf-8') if isinstance(f.numpy(), bytes) else f.numpy().decode('utf-8') for f in fb]
    if store_probs:
        for i in range(yp.shape[0]):
            out_name = f"prob_{len(all_files_for_sample)}.npy"
            np.save(os.path.join(probs_dir, out_name), yp[i].numpy().astype(np.float32))

elapsed = time.time() - start

Yp = np.concatenate(all_probs, axis=0)
Yt = np.concatenate(all_trues, axis=0)
print('Inference done:', Yp.shape, 'time(s)=', round(elapsed, 2))

In [ ]:
# Metrics: micro-averaged IoU, Dice, Precision, Recall, F1; threshold sweep
import csv

# Squeeze last channel for Yt/Yp comparison
if Yp.ndim == 4 and Yp.shape[-1] == 1:
    Yp_eval = Yp[..., 0]
else:
    Yp_eval = Yp
if Yt.ndim == 4 and Yt.shape[-1] == 1:
    Yt_eval = Yt[..., 0]
else:
    Yt_eval = Yt

THRESHOLDS = np.linspace(0.05, 0.90, 18)
rows = []
for t in THRESHOLDS:
    yp_bin = (Yp_eval >= t).astype(np.uint8)
    yt_bin = (Yt_eval >= 0.5).astype(np.uint8)
    inter = np.logical_and(yp_bin==1, yt_bin==1).sum()
    union = np.logical_or(yp_bin==1, yt_bin==1).sum()
    iou = inter / max(1, union)
    sum_p = yp_bin.sum(); sum_t = yt_bin.sum()
    dice = 2*inter / max(1, sum_p + sum_t)
    tp = inter
    fp = np.logical_and(yp_bin==1, yt_bin==0).sum()
    fn = np.logical_and(yp_bin==0, yt_bin==1).sum()
    prec = tp / max(1, (tp+fp))
    rec = tp / max(1, (tp+fn))
    f1 = 2*prec*rec / max(1e-9, (prec+rec))
    rows.append({'threshold': float(t), 'iou': float(iou), 'dice': float(dice), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1)})

best = max(rows, key=lambda r: r['f1'])
print('Best by F1:', best)

sweep_csv = os.path.join(RESULTS_DIR, 'threshold_sweep_test.csv')
with open(sweep_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print('Wrote sweep to', sweep_csv)

In [ ]:
# Per-file and global metrics aggregation; export CSVs
import csv

best_t = float(best['threshold'])
yp_bin_best = (Yp_eval >= best_t).astype(np.uint8)
yt_bin = (Yt_eval >= 0.5).astype(np.uint8)

# Build per-sample metrics
per_sample_rows = []
for i in range(Yt_eval.shape[0]):
    inter = np.logical_and(yp_bin_best[i]==1, yt_bin[i]==1).sum()
    union = np.logical_or(yp_bin_best[i]==1, yt_bin[i]==1).sum()
    iou = inter / max(1, union)
    sp = yp_bin_best[i].sum(); st = yt_bin[i].sum()
    dice = 2*inter / max(1, sp + st)
    per_sample_rows.append({'index': i, 'file': all_files_for_sample[i], 'iou': float(iou), 'dice': float(dice)})

# Aggregate per-file means
from collections import defaultdict
acc = defaultdict(lambda: {'iou': [], 'dice': []})
for r in per_sample_rows:
    acc[r['file']]['iou'].append(r['iou'])
    acc[r['file']]['dice'].append(r['dice'])

per_file_rows = []
for fp, vals in acc.items():
    per_file_rows.append({'file': fp, 'mean_iou': float(np.mean(vals['iou'])), 'mean_dice': float(np.mean(vals['dice'])), 'n': len(vals['iou'])})

summary = {
    'n_samples': int(Yt_eval.shape[0]),
    'best_threshold': best_t,
    'global_iou': float((np.logical_and(yp_bin_best==1, yt_bin==1).sum()) / max(1, np.logical_or(yp_bin_best==1, yt_bin==1).sum())),
    'global_dice': float(2*np.logical_and(yp_bin_best==1, yt_bin==1).sum() / max(1, yp_bin_best.sum() + yt_bin.sum())),
    'mean_pred_max': float(np.mean(np.max(Yp_eval, axis=(1,2))))
}
print('Summary:', summary)

# Write CSVs
per_sample_csv = os.path.join(RESULTS_DIR, 'per_sample.csv')
per_file_csv = os.path.join(RESULTS_DIR, 'per_file.csv')
summary_csv = os.path.join(RESULTS_DIR, 'summary.csv')

with open(per_sample_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(per_sample_rows[0].keys()))
    w.writeheader(); w.writerows(per_sample_rows)
with open(per_file_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(per_file_rows[0].keys()))
    w.writeheader(); w.writerows(per_file_rows)
with open(summary_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(summary.keys()))
    w.writeheader(); w.writerow(summary)

print('Wrote:', per_sample_csv, per_file_csv, summary_csv)

In [ ]:
# Qualitative plots (paper-style) for a few samples
import matplotlib.pyplot as plt

# pick first 3 samples
n_show = min(3, Yt_eval.shape[0])
for i in range(n_show):
    pred = Yp_eval[i]
    pred_bin = (pred >= best_t).astype(np.uint8)
    ku_dbz = ( (Yp_eval[i]*0) + 1 )  # placeholder if needed
    # Try to reconstruct Ku from input probabilities is not possible here.
    # Instead, rebuild for the originating file to get reflectivity and mask for the same scan index.
    fp = all_files_for_sample[i]
    X_vis, Y_vis = dgx.build_arrays_for_mask_mode([fp],
                                                  mask_mode=MASK_MODE,
                                                  use_z_final=USE_Z_FINAL,
                                                  use_dfr_ml=USE_DFR_ML,
                                                  include_latlon=INCLUDE_LATLON,
                                                  nbin_target=N_BIN_TARGET,
                                                  min_val=MIN_VAL,
                                                  max_val=MAX_VAL,
                                                  max_scans=1)
    ku_dbz = X_vis[0, ..., 0] * (MAX_VAL - MIN_VAL) + MIN_VAL
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    im0 = axs[0].imshow(ku_dbz, origin='lower', aspect='auto', cmap='RdYlGn_r', vmin=MIN_VAL, vmax=MAX_VAL)
    axs[0].set_title('Ku Reflectivity (dBZ)'); plt.colorbar(im0, ax=axs[0])
    im1 = axs[1].imshow(pred, origin='lower', aspect='auto', cmap='hot', vmin=0, vmax=1)
    axs[1].set_title(f'Prediction (max={pred.max():.2f})'); plt.colorbar(im1, ax=axs[1])
    im2 = axs[2].imshow(pred_bin, origin='lower', aspect='auto', cmap='gray_r', vmin=0, vmax=1)
    axs[2].set_title(f'Binary ≥ {best_t:.2f}')
    plt.tight_layout()
    outp = os.path.join(FIG_DIR, f'test_sample_{i}.png')
    plt.savefig(outp, dpi=200); plt.show()
    print('Saved', outp)

In [ ]:
# Throughput/latency benchmark
import time, json

bench = {}
for bs in [4, 8, 16]:
    ds = make_test_dataset(test_files[:max(1, len(test_files)//4)], batch_size=bs)
    # warmup
    for xb, yb, fb in ds.take(2):
        _ = model.predict(xb, verbose=0)
    # measure
    iters, nimg, t0 = 10, 0, time.time()
    for xb, yb, fb in ds.take(iters):
        nimg += xb.shape[0]
        _ = model.predict(xb, verbose=0)
    dt = time.time() - t0
    bench[str(bs)] = {
        'batches': iters,
        'samples': int(nimg),
        'elapsed_s': float(dt),
        'imgs_per_s': float(nimg/dt) if dt > 0 else None,
        'avg_latency_s': float(dt/iters)
    }

bench_json = os.path.join(RESULTS_DIR, 'benchmark.json')
with open(bench_json, 'w', encoding='utf-8') as f:
    json.dump(bench, f, indent=2)
print('Benchmark saved to', bench_json)
bench

In [ ]:
# Optional: export ONNX and verify (requires tf2onnx)
try:
    import tf2onnx
    import onnx
    tmp_saved = 'tmp_saved_model_for_onnx'
    model.save(tmp_saved)
    onnx_model, _ = tf2onnx.convert.from_saved_model(tmp_saved, output_path=None)
    onnx_path = os.path.join(RESULTS_DIR, 'model.onnx')
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print('Exported ONNX to', onnx_path)
except Exception as e:
    print('ONNX export skipped:', e)